In [1]:
import json
from collections import Counter
from pathlib import Path

In [2]:
PRED_PATH = "/lambda/nfs/neel/Research/runs/blip2/vqa/vqa_v2_balanced_5k/preds.jsonl"
OUT_PATH  = "/lambda/nfs/neel/Research/runs/blip2/vqa/vqa_v2_balanced_5k/metrics/vqav2_metrics.json"


In [3]:
def norm(s): 
    return str(s).strip().lower()

def vqa_soft_acc_from_answers(pred, answers):
    pred = norm(pred)
    counts = Counter(norm(a) for a in answers)
    return min(counts.get(pred, 0) / 3.0, 1.0)

total = 0
sum_logged = 0.0
sum_recomputed = 0.0
mismatch = 0

In [4]:
with open(PRED_PATH, "r") as f:
    for line in f:
        ex = json.loads(line)
        pred = ex["prediction"]
        answers = ex["gt_answers"]

        logged = float(ex.get("soft_acc", vqa_soft_acc_from_answers(pred, answers)))
        recomputed = float(vqa_soft_acc_from_answers(pred, answers))

        sum_logged += logged
        sum_recomputed += recomputed
        total += 1

        if abs(logged - recomputed) > 1e-6:
            mismatch += 1

avg_logged = sum_logged / total
avg_recomputed = sum_recomputed / total

In [5]:
metrics = {
    "n": total,
    "vqa_soft_acc_logged_mean": avg_logged,
    "vqa_soft_acc_recomputed_mean": avg_recomputed,
    "num_soft_acc_mismatches": mismatch,
}

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(metrics)
print("Saved:", OUT_PATH)

{'n': 5000, 'vqa_soft_acc_logged_mean': 0.29319999999999974, 'vqa_soft_acc_recomputed_mean': 0.29319999999999974, 'num_soft_acc_mismatches': 0}
Saved: /lambda/nfs/neel/Research/runs/blip2/vqa/vqa_v2_balanced_5k/metrics/vqav2_metrics.json
